# 🧠 Skill2Income AI - Análise Exploratória e Tokenização (EDA)

Este notebook executa o passo fundamental de engenharia de dados do framework: a ingestão de perfis textuais brutos, higienização de strings e validação volumétrica de tokens utilizando o codificador oficial da OpenAI (`tiktoken`).

**Objetivos:**
1. Carregar a massa de dados bruta simulada (`data/raw/user_input_sample.json`).
2. Higienizar e sanitizar os fluxos de texto (Remoção de quebras de linha e ruídos).
3. Avaliar a volumetria de tokens para garantir o encaixe na janela de contexto (*Context Window*) e estimar custos operacionais da API.

In [ ]:
# Instalação das dependências necessárias para análise local
!pip install pandas tiktoken

In [ ]:
import json
import os
import pandas as pd
import tiktoken

## 1. Ingestão e Leitura dos Dados Brutos

In [ ]:
# Navega a partir da pasta de notebooks para a raiz de dados
raw_data_path = "../data/raw/user_input_sample.json"

with open(raw_data_path, "r", encoding="utf-8") as file:
    payload_bruto = json.load(file)

print(f"Usuário Identificado: {payload_bruto['usuario_id']}")
print(f"Fonte dos Dados: {payload_bruto['fonte_dados']}")
print("-" * 50)
print("Amostra do Texto Capturado:")
print(payload_bruto["raw_text"][:350] + "...")

## 2. Higienização e Tratamento do Texto

Removemos quebras de linha duplicadas, espaços residuais e caracteres ocultos comuns em extrações de currículos em formato PDF.

In [ ]:
def limpar_texto_perfil(texto: str) -> str:
    if not texto:
        return ""
    # Remove caracteres nulos e normaliza espaçamentos
    texto_limpo = texto.replace("\n", " ").replace("\r", " ")
    texto_limpo = " ".join(texto_limpo.split())
    return texto_limpo

texto_processado = limpar_texto_perfil(payload_bruto["raw_text"])
print(f"Comprimento do texto bruto: {len(payload_bruto['raw_text'])} caracteres")
print(f"Comprimento do texto limpo: {len(texto_processado)} caracteres")

## 3. Análise Volumétrica de Tokens (Metrificação OpenAI)

Modelos como o `gpt-4o-mini` utilizam a codificação **cl100k_base** ou **o200k_base**. Vamos medir o volume exato de tokens consumido pelo perfil para garantir eficiência de custo e performance de rede.

In [ ]:
# Inicializa o codificador padrão para modelos GPT-4
encoder = tiktoken.get_encoding("cl100k_base")

tokens_brutos = encoder.encode(payload_bruto["raw_text"])
tokens_limpos = encoder.encode(texto_processado)

print(f"Total de Tokens (Texto Bruto): {len(tokens_brutos)}")
print(f"Total de Tokens (Texto Limpo): {len(tokens_limpos)}")

# Estimativa de Custo de Ingestão baseado em tabelas padrão (Ex: $0.15 por 1M tokens)
custo_estimado = (len(tokens_limpos) / 1_000_000) * 0.15
print(f"Custo de Ingestão da Célula de IA: ${custo_estimado:.7f} USD")

## 4. Conclusão da Análise

O perfil possui volumetria controlada (~400 a 600 tokens), o que se enquadra perfeitamente na janela de contexto padrão da API da OpenAI. Os dados limpos estão prontos para serem encaminhados de forma segura à esteira do `SkillExtractor` no microsserviço principal.